In [ ]:
import kfp
from google.cloud import aiplatform
from kfp.v2 import dsl, compiler
from kfp.v2.dsl import component
from google.cloud import storage
import os
from datetime import datetime
from uuid import uuid4
import pytz
from typing import NamedTuple
from dotenv import load_dotenv 
import json
from google_cloud_pipeline_components.v1.vertex_notification_email import VertexNotificationEmailOp

In [50]:
# Definiendo ruta de ejecucion del proyecto
PATH_ROOT = os.getcwd()
post_project = [i for i, val in enumerate(str(PATH_ROOT).split('\\')) if val == 'project-data-processing'][0]
os.chdir('\\'.join(str(PATH_ROOT).split('\\')[:post_project + 1]))

def generar_run_id() -> str:
    timestamp = datetime.now(pytz.utc).strftime("%Y%m%dT%H%M%SZ")
    token = uuid4().hex[:12]
    return f"house-price-{timestamp}-{token}"

In [51]:
AAAAMM = '202606'
run_id = generar_run_id()

In [52]:
load_dotenv()

pipeline_config = {

    "GCP_PROJECT_ID": os.environ["GCP_PROJECT_ID"],
    "GCP_BUCKET_NAME": os.environ["GCP_BUCKET_NAME"],
    "GCP_MODEL_PROJECT": os.environ["GCP_MODEL_PROJECT"],
    "MODEL_ROOT": os.environ["MODEL_ROOT"],
    "PIPELINE_ROOT": os.environ["PIPELINE_ROOT"],

    # Tables inputs
    "GCP_PROJECT_ID_INPUT": os.environ["GCP_PROJECT_ID_INPUT"],
    "BQ_DATASET_ID_INPUT": os.environ["BQ_DATASET_ID_INPUT"],
    "BQ_TABLE_ID_INPUT": os.environ["BQ_TABLE_ID_INPUT"],

    # Tables features
    "BQ_DATASET_ID_FEATURES": os.environ["BQ_DATASET_ID_FEATURES"],
    "BQ_TABLE_ID_FEATURES": os.environ["BQ_TABLE_ID_FEATURES"],

    # Tables temporales
    "BQ_DATASET_ID_TEMP": os.environ["BQ_DATASET_ID_TEMP"],
    "BQ_TABLE_ID_TEMP_DATA": os.environ["BQ_TABLE_ID_TEMP_DATA"],
    "BQ_TABLE_ID_TEMP_DATA_TRANSF": os.environ["BQ_TABLE_ID_TEMP_DATA_TRANSF"],
    "BQ_TABLE_ID_TEMP_DATA_PREDICT": os.environ["BQ_TABLE_ID_TEMP_DATA_PREDICT"],

    # Tablas Outputs
    "GCP_PROJECT_ID_OUT": os.environ["GCP_PROJECT_ID_OUT"],
    "BQ_DATASET_ID_OUT": os.environ["BQ_DATASET_ID_OUT"],
    "BQ_TABLE_ID_OUT": os.environ["BQ_TABLE_ID_OUT"],
    "BQ_TABLE_ID_OUT_HIST": os.environ["BQ_TABLE_ID_OUT_HIST"],

    # GSA
    "GSA_NAME":os.environ["GSA_NAME"] 
}

bucket_name = pipeline_config["GCP_BUCKET_NAME"]
model_project = pipeline_config["GCP_MODEL_PROJECT"]
project_id = pipeline_config["GCP_PROJECT_ID"]

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

### Componente 1: Filter Input

In [53]:
# Componente 1:
@component(
    base_image="python:3.9-slim",
    packages_to_install=[
        "pyarrow==9.0.0",
        "numpy==1.26.4",
        "pandas==1.5.3",
        "google-cloud-bigquery==2.34.4",
        "pytz"
    ],
)
def filter_input(
    periodo: str, 
    run_id: str,
    config_json: str
) -> str:
    
    import os
    from google.cloud import bigquery
    from datetime import datetime
    import pytz
    from google.api_core.exceptions import NotFound
    import pandas as pd
    import json

    config = json.loads(config_json)

    # Name Project
    project_id = config["GCP_PROJECT_ID"]

    # Tablas BigQuery inputs
    project_id_input = config["GCP_PROJECT_ID_INPUT"]
    dataset_id_input = config["BQ_DATASET_ID_INPUT"]
    table_id_input = config["BQ_TABLE_ID_INPUT"]

    # Tablas features
    dataset_id_features = config["BQ_DATASET_ID_FEATURES"]
    table_id_features = config["BQ_TABLE_ID_FEATURES"]

    # Tablas temporales
    dataset_temp = config["BQ_DATASET_ID_TEMP"]
    table_temp_data_input = config["BQ_TABLE_ID_TEMP_DATA"]


    # Carga de BigQuery
    client = bigquery.Client(project=project_id)

    path_bq_table = f"{project_id_input}.{dataset_id_input}.{table_id_input}"

    dfInput = client.query(
            f'''SELECT * EXCEPT(date_subida_local,date_subida_utc)
            FROM `{path_bq_table}` where periodo = '{periodo}'
            '''
        ).to_dataframe()

    path_bq_table_feature = f"{project_id}.{dataset_id_features}.{table_id_features}"
    dfFeatures = client.query(
            f'''SELECT * FROM `{path_bq_table_feature}`
            '''
        ).to_dataframe()

    listFeatures = dfFeatures['features'].tolist()

    dfInputFeat = dfInput[['id','periodo'] + listFeatures].copy()

    user_id = client.query("SELECT SESSION_USER()").to_dataframe().iloc[0, 0]
    fecha_carga = datetime.now(pytz.timezone("America/Lima"))

    dfInputFeat['run_id'] = run_id
    dfInputFeat['creation_user'] = user_id
    dfInputFeat['str_process_date_local'] = fecha_carga.replace(tzinfo=None).strftime('%Y%m%d')
    dfInputFeat['process_datetime_local'] = fecha_carga.replace(tzinfo=None)
    dfInputFeat['process_datetime_utc'] = fecha_carga.astimezone(pytz.UTC)


    dfInputFeat = dfInputFeat[  ['id','periodo'] 
                            + listFeatures 
                            + ['run_id','creation_user','str_process_date_local','process_datetime_local','process_datetime_utc']]
    periodo_out = str(dfInputFeat['periodo'].iloc[0])

    # Validacion de duplicados: 
    dfVAlDup = dfInputFeat.groupby(['id','periodo']).size().reset_index().rename(columns = {0:'cant'})
    Res = dfVAlDup[dfVAlDup['cant'] > 1]

    if Res.shape[0] > 0:
        print(f"Error de duplicados: {Res.shape[0]} casos")
        print(Res)
        exit()

    path_bq_data_input = f"{project_id}.{dataset_temp}.{table_temp_data_input}"
    try:
        delete_query = f"""
            DELETE FROM `{path_bq_data_input}`
            WHERE periodo = '{periodo}'
        """
        delete_job = client.query(delete_query)
        delete_job.result()
        print(f"Registros previos eliminados para período {periodo}.")

    except NotFound:
        print("La tabla destino no existe aún; se creará durante la carga.")

    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_APPEND"
    )
    job = client.load_table_from_dataframe(
        dfInputFeat,
        path_bq_data_input,
        job_config=job_config
    )
    job.result()
    print(f"Tabla cargada: {path_bq_data_input}") 

    return periodo_out

### Componente 2: Transformt data

In [54]:
@component(
    base_image="python:3.9-slim",    
    packages_to_install=[
        "numpy==1.26.4",
        "pyarrow==9.0.0",
        "pandas==1.5.3",
        "google-cloud-storage",
        "google-cloud-bigquery==2.34.4",
        "scikit-learn==1.4.2",
        "joblib",
        "pytz"
    ],
)
def transform_data(
    periodo: str,
    run_id: str,
    config_json: str
) -> NamedTuple(
    "TransformOutputs",
[
        ("periodo_out", str),
        ("pipeline_name", str),
],
):

    import os
    from google.cloud import storage
    from google.cloud import bigquery
    from google.api_core.exceptions import NotFound
    import types
    import joblib
    import sys
    from io import BytesIO
    from datetime import datetime
    import pandas as pd
    import pytz
    import json

    config = json.loads(config_json)

    # Name Project
    project_id = config["GCP_PROJECT_ID"]

    # Ruta Storage
    pipeline_ruta = config["PIPELINE_ROOT"]
    bucket_name = config["GCP_BUCKET_NAME"]

    # Tablas temporales
    dataset_temp = config["BQ_DATASET_ID_TEMP"]
    table_temp_data_input = config["BQ_TABLE_ID_TEMP_DATA"]
    table_temp_data_transf_input = config["BQ_TABLE_ID_TEMP_DATA_TRANSF"]


    # Cliente de Cloud Storage
    client = storage.Client(project=project_id)
    bucket = client.bucket(bucket_name)

    metadata_name = 'metadata_transformer.py'
    metadata_path = f"{pipeline_ruta}/{metadata_name}"
    blob = bucket.blob(metadata_path)
    module_code  = blob.download_as_text(encoding="utf-8")
    print("Archivo PY cargado correctamente ✅")
    print(type(module_code))
    metadata_transformer = types.ModuleType('metadata_transformer')
    metadata_transformer.__file__ = "gs://.../metadata_transformer.py"
    sys.modules['metadata_transformer'] = metadata_transformer
    exec(compile(module_code, metadata_transformer.__file__, "exec"),
        metadata_transformer.__dict__)

    pipeline_name = 'Pipeline-Transformacion-Training.joblib'
    pipeline_path = f"{pipeline_ruta}/{pipeline_name}"
    blob = bucket.blob(pipeline_path)
    contenido_joblib  = blob.download_as_bytes()
    pipeline = joblib.load(BytesIO(contenido_joblib))
    print("Archivo JOBLIB cargado correctamente ✅")
    print(type(pipeline))


    # Cliente BigQuery
    client = bigquery.Client(project=project_id)
    path_bq_data_input = f"{project_id}.{dataset_temp}.{table_temp_data_input}"
    dfInputFeat = client.query(
            f'''SELECT * EXCEPT(run_id, creation_user, str_process_date_local, process_datetime_local, process_datetime_utc)
            FROM `{path_bq_data_input}` where periodo = '{periodo}'
            '''
    ).to_dataframe()

    X_escalado = pipeline.transform(dfInputFeat)
    feature_order = pipeline.named_steps[
        "feature_engineering"
    ].get_feature_names_out()
    col_orden = [col + '_transf' for col in feature_order]

    dfInputFeatTransf = pd.DataFrame(
        X_escalado,
        columns = col_orden,
        index=  dfInputFeat.index,
    )

    user_id = client.query("SELECT SESSION_USER()").to_dataframe().iloc[0, 0]
    fecha_carga = datetime.now(pytz.timezone("America/Lima"))
    periodo_out = str(dfInputFeat.periodo[0])

    dfInputFeatTransf['id'] = dfInputFeat['id']
    dfInputFeatTransf['run_id'] = run_id
    dfInputFeatTransf['creation_user'] = user_id
    dfInputFeatTransf['str_process_date_local'] = fecha_carga.replace(tzinfo=None).strftime('%Y%m%d')
    dfInputFeatTransf['process_datetime_local'] = fecha_carga.replace(tzinfo=None)
    dfInputFeatTransf['process_datetime_utc'] = fecha_carga.astimezone(pytz.UTC)
    dfInputFeatTransf['periodo'] = periodo_out

    dfInputFeatTransf = dfInputFeatTransf[
                                        ['id','periodo'] 
                                        + col_orden 
                                        + ['run_id','creation_user','str_process_date_local','process_datetime_local','process_datetime_utc']
                    ]

    path_bq_data_transf_input = f"{project_id}.{dataset_temp}.{table_temp_data_transf_input}"
    try:
        delete_query = f"""
            DELETE FROM `{path_bq_data_transf_input}`
            WHERE periodo = '{periodo}'
        """
        delete_job = client.query(delete_query)
        delete_job.result()
        print(f"Registros previos eliminados para período {periodo}.")

    except NotFound:
        print("La tabla destino no existe aún; se creará durante la carga.")

    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_APPEND"
    )
    job = client.load_table_from_dataframe(
        dfInputFeatTransf,
        path_bq_data_transf_input,
        job_config=job_config
    )
    job.result()
    print(f"Tabla cargada: {path_bq_data_transf_input}")

    return periodo_out, pipeline_name

### Componente 3: Process Predict

In [55]:
@component(
    base_image="python:3.9-slim",    
    packages_to_install=[
        "numpy==1.26.4",
        "pyarrow==9.0.0",
        "pandas==1.5.3",
        "google-cloud-storage",
        "google-cloud-bigquery==2.34.4",
        "scikit-learn==1.4.2",
        "pytz"
    ],
)
def process_predict(
    periodo: str,
    run_id: str,
    pipeline_name: str,
    config_json: str
) -> str:

    import os
    from google.cloud import storage
    from google.cloud import bigquery
    from google.api_core.exceptions import NotFound
    import pandas as pd
    import joblib
    from io import BytesIO
    from datetime import datetime
    import pytz
    import json
    
    config = json.loads(config_json)
    
    # Name Project
    project_id = config["GCP_PROJECT_ID"]
    bucket_name = config["GCP_BUCKET_NAME"]
    model_ruta = config["MODEL_ROOT"]

    # Tablas temporales
    dataset_temp = config["BQ_DATASET_ID_TEMP"]
    table_temp_data_transf_input = config["BQ_TABLE_ID_TEMP_DATA_TRANSF"]
    table_temp_data_predict = config["BQ_TABLE_ID_TEMP_DATA_PREDICT"]


    model_name  = "Model-GradientBoostingRegressor.joblib"
    model_path = f"{model_ruta}/{model_name}"

    # Cliente de Cloud Storage
    client = storage.Client(project=project_id)
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(model_path)
    contenido_joblib  = blob.download_as_bytes()
    modelo = joblib.load(BytesIO(contenido_joblib))
    print("Archivo JOBLIB cargado correctamente ✅")
    print(type(modelo))

    # Cliente BigQuery
    client = bigquery.Client(project=project_id)

    path_bq_data_transf_input = f"{project_id}.{dataset_temp}.{table_temp_data_transf_input}"
    dfInputFeatTransf = client.query(
            f'''SELECT * EXCEPT(run_id, creation_user, str_process_date_local, process_datetime_local, process_datetime_utc)
            FROM `{path_bq_data_transf_input}` where periodo = '{periodo}'
            '''
    ).to_dataframe()
    dfInputFeatTransf['saleprice_predict'] = modelo.predict(dfInputFeatTransf.drop(columns = ['id','periodo']))
    dfPredict = dfInputFeatTransf[['id','saleprice_predict']].copy()

    user_id = client.query("SELECT SESSION_USER()").to_dataframe().iloc[0, 0]
    fecha_carga = datetime.now(pytz.timezone("America/Lima"))
    periodo_out = str(dfInputFeatTransf.periodo[0])
    model_name = model_path.split('/')[-1]

    dfPredict['run_id'] = run_id
    dfPredict['creation_user'] = user_id
    dfPredict['str_process_date_local'] = fecha_carga.replace(tzinfo=None).strftime('%Y%m%d')
    dfPredict['process_datetime_local'] = fecha_carga.replace(tzinfo=None)
    dfPredict['process_datetime_utc'] = fecha_carga.astimezone(pytz.UTC)
    dfPredict['periodo'] = periodo_out
    dfPredict['model_name'] = model_name

    dfPredict =  dfPredict[
            ['id','periodo','saleprice_predict','model_name','run_id','creation_user','str_process_date_local',
            'process_datetime_local','process_datetime_utc']
    ].copy()

    path_bq_data_predict = f"{project_id}.{dataset_temp}.{table_temp_data_predict}"
    try:
        delete_query = f"""
            DELETE FROM `{path_bq_data_predict}`
            WHERE periodo = '{periodo}'
        """
        delete_job = client.query(delete_query)
        delete_job.result()
        print(f"Registros previos eliminados para período {periodo}.")

    except NotFound:
        print("La tabla destino no existe aún; se creará durante la carga.")

    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_APPEND"
    )
    job = client.load_table_from_dataframe(
        dfPredict,
        path_bq_data_predict,
        job_config=job_config
    )
    job.result()
    print(f"Tabla cargada: {path_bq_data_predict}") 

    table_id_auditoria = f"{project_id}.{dataset_temp}.model_execution_audit"
    user_id = client.query("SELECT SESSION_USER()").to_dataframe().iloc[0, 0]
    fecha_carga = datetime.now(pytz.timezone("America/Lima"))
    df_auditoria = pd.DataFrame([{
        "periodo": periodo_out,
        "run_id" : run_id,
        "model_name": model_name,
        "model_path_gcs": f"gs://{bucket_name}/{model_path}",
        "pipeline_path_gcs": (
            f"gs://{bucket_name}/{model_ruta}/{pipeline_name}"
        ),
        "rows_processed": len(dfPredict),
        "str_process_date_local": fecha_carga.replace(tzinfo=None).strftime('%Y%m%d'),
        "process_datetime_local": fecha_carga.replace(tzinfo=None),
        "process_datetime_utc": fecha_carga.astimezone(pytz.UTC)
    }])
    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_APPEND"
    )
    job = client.load_table_from_dataframe(
        df_auditoria,
        table_id_auditoria,
        job_config=job_config
    )
    job.result()
    print(f"Auditoría registrada en: {table_id_auditoria}")

    return periodo_out


### Componente 4: Send out

In [56]:
@component(
    base_image="python:3.9-slim",        
    packages_to_install=[
        "numpy==1.26.4",
        "pyarrow==9.0.0",
        "pandas==1.5.3",
        "google-cloud-bigquery==2.34.4",
        "pytz"
    ],
)
def send_out(
    periodo: str,
    config_json: str
):

    import os
    from google.cloud import bigquery
    from google.api_core.exceptions import NotFound
    import pandas as pd
    import pytz
    from datetime import datetime
    import json

    config = json.loads(config_json)
    
    # Name Project
    project_id = config["GCP_PROJECT_ID"]

    # Tablas Ouputs
    project_id_out = config["GCP_PROJECT_ID_OUT"]
    dataset_id_out = config["BQ_DATASET_ID_OUT"]
    table_out = config["BQ_TABLE_ID_OUT"]
    table_out_hist = config["BQ_TABLE_ID_OUT_HIST"]


    # Cliente BigQuery
    client = bigquery.Client(project=project_id)

    query = f"""
    CALL `gcp-processing-vertex-prod-us.dev_table.sp_table_out_model` ('{periodo}')
    """
    job = client.query(query)
    dfOut = job.result().to_dataframe()
    fecha_carga = datetime.now(pytz.timezone("America/Lima"))
    dfOut['str_process_date_local'] = fecha_carga.replace(tzinfo=None).strftime('%Y%m%d')
    dfOut['process_datetime_local'] = fecha_carga.replace(tzinfo=None)
    dfOut['process_datetime_utc'] = fecha_carga.astimezone(pytz.UTC)

    table_id_out_hist = f'{project_id_out}.{dataset_id_out}.{table_out_hist}'
    try:
        delete_query = f"""
            DELETE FROM `{table_id_out_hist}`
            WHERE periodo = '{periodo}'
        """
        delete_job = client.query(delete_query)
        delete_job.result()
        print(f"Registros previos eliminados para período {periodo}.")

    except NotFound:
        print("La tabla destino no existe aún; se creará durante la carga.")

    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_APPEND"
    )
    job = client.load_table_from_dataframe(
        dfOut,
        table_id_out_hist,
        job_config=job_config
    )
    job.result()
    print(f"Subida a PRD registrada en: {table_id_out_hist}")

    table_id_out = f'{project_id_out}.{dataset_id_out}.{table_out}'
    try:
        delete_query = f"""
            DELETE FROM `{table_id_out}`
            WHERE periodo = '{periodo}'
        """
        delete_job = client.query(delete_query)
        delete_job.result()
        print(f"Registros previos eliminados para período {periodo}.")

    except NotFound:
        print("La tabla destino no existe aún; se creará durante la carga.")

    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE"
    )
    job = client.load_table_from_dataframe(
        dfOut,
        table_id_out,
        job_config=job_config
    )
    job.result()
    print(f"Subida a PRD registrada en: {table_id_out}")


## Pipeline

In [ ]:
@kfp.dsl.pipeline(
    name="pipeline-data-processing-predict", 
    description="Proyecto de pipeline para data processing y predict",
    pipeline_root=f"gs://{bucket_name}/{model_project}"
)
def main_pipeline(
    AAAAMM: str, 
    run_id: str,
    config_json_t: str
    
):

    notification_email_task = VertexNotificationEmailOp(
        recipients = ["mori.brenis.junior@gmail.com"]
    )
    notification_email_task.set_display_name("NOTIFICATION_EMAIL")

    with dsl.ExitHandler(notification_email_task, name="Execute pipeline prediction"):

        filter_task = filter_input(
            periodo = AAAAMM,
            run_id = run_id,
            config_json=config_json_t
        )
        filter_task.set_display_name("FILTER DATA INPUT")

        transform_task = transform_data(
            periodo = filter_task.output,
            run_id = run_id,
            config_json=config_json_t        
        ).after(filter_task)
        transform_task.set_display_name("TRANSFORM DATA")

        predict_task = process_predict(
            periodo = transform_task.outputs["periodo_out"],
            run_id = run_id,
            pipeline_name = transform_task.outputs["pipeline_name"],
            config_json=config_json_t
        ).after(transform_task)
        predict_task.set_display_name("PROCESS PREDICT")

        out_task = send_out(
            periodo = predict_task.output,
            config_json=config_json_t
        )
        out_task.set_display_name("SEND OUT")


In [58]:
compiler.Compiler().compile(
    pipeline_func=main_pipeline,
    package_path="pipeline-processing-predict.json"
)

d:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\kfp\v2\compiler\compiler.py:1290: FutureWarning: APIs imported from the v1 namespace (e.g. kfp.dsl, kfp.components, etc) will not be supported by the v2 compiler since v2.0.0
  warnings.warn(


In [59]:
def upload_to_gcs(project_id, bucket_name, source_file_name, destination_blob_name):
    storage_client = storage.Client(project = project_id)
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(source_file_name)
    print(f"Archivo {source_file_name} subido a {destination_blob_name} en el bucket {bucket_name}.")


destination_blob_name = f"{model_project}/pipeline_end_to_end/pipeline-processing-predict.json"
pipeline_path = "pipeline-processing-predict.json"
upload_to_gcs(project_id, bucket_name, pipeline_path, destination_blob_name)

Archivo pipeline-processing-predict.json subido a project-pipeline-predictions-casas/pipeline_end_to_end/pipeline-processing-predict.json en el bucket gcp-processing-storage-prod.


In [60]:
aiplatform.init(project=project_id, location="us-central1")

job = aiplatform.PipelineJob(
    display_name="pipeline-data-processing-predict",
    template_path=f"gs://{bucket_name}/{destination_blob_name}",
    enable_caching=False,
    project=project_id,
    location="us-central1",
    parameter_values={
                        "AAAAMM":AAAAMM,
                        "run_id":run_id,
                        "config_json_t":pipeline_config
                     },
    labels={"module": "ml", "application": "app", "chapter": "mlops", "environment": "prd", "owner": "jmorib"}
)

print('submit pipeline job ...')
job.submit(service_account=pipeline_config["GSA_NAME"])

submit pipeline job ...
Creating PipelineJob
PipelineJob created. Resource name: projects/911699346669/locations/us-central1/pipelineJobs/pipeline-data-processing-predict-20260808212029
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/911699346669/locations/us-central1/pipelineJobs/pipeline-data-processing-predict-20260808212029')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/pipeline-data-processing-predict-20260808212029?project=911699346669
